In [0]:
import logging
import pyspark.sql.functions as F
from databricks.feature_engineering import FeatureEngineeringClient

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

In [0]:
fe = FeatureEngineeringClient()

In [0]:
catalog_name = "supply_chain_daily_demand"
gold_schema_name = "gold"
featured_gold_table_name = "features_daily_demand"
feature_schema = "features"
feature_table_name = "daily_demand_features"
label_table_name = "daily_demand_label"

In [0]:
class FeatureStore:
    def read_gold_table(self):
        gold_table_path = f"{catalog_name}.{gold_schema_name}.{featured_gold_table_name}"

        logging.info(f"Loading the feature table from the {gold_table_path}")
        self.data = spark.read.table(gold_table_path)
        logging.info(f"Gold feature table loaded in the dataframe.")

    def add_primary_key(self):
        logging.info("Adding primary key: feature_uid")
        self.data = self.data.withColumn("feature_uid", F.concat(F.col("transaction_date").cast("string"),F.lit('_'), F.col("product_name"),F.lit('_'),F.col("destination_city")))
        logging.info("Primary key is added. Key: feature_uid")

    def register_feature_table(self):
        feature_table_path = f"{catalog_name}.{feature_schema}.{feature_table_name}"
        
        logging.info("Selected the req columns for the feature table")
        feature_data = self.data.select(["feature_uid", "transaction_date", "product_name", "destination_city", "avg_unit_price", "total_inventory", "day_of_week", "month", "demand_lag_1", "demand_lag_7", "moving_avg_7"])

        if not spark.catalog.tableExists(feature_table_path):
            tags = {'name':"daily_demand_features", 'domain':"oil&gas", 'key_columns':"feature_uid, transaction_date, product_name, destination_city","categorical_columns":"product_name, destination_city", "numeric_cols":"total_inventory,day_of_week,month,demand_lag_1,demand_lag_7,moving_avg_7"}
            
            schema = feature_data.schema

            logging.info(f"Creating the Feature table at {feature_table_path}")
            fe.create_table(name=feature_table_path, primary_keys=["feature_uid"], schema=schema, description="Raw copy of the Feature data", tags=tags)
            logging.info("Feature table is created.")

        logging.info(f"Inserting the table with the data. Table: {feature_table_path}")
        fe.write_table(name=feature_table_path, df=feature_data, mode="merge")
        logging.info(f"Data Inserted Table: {feature_table_path}")

    def save_label_data(self):
        label_table_path = f"{catalog_name}.{gold_schema_name}.{label_table_name}"
        gold_table_path = f"{catalog_name}.{gold_schema_name}.{featured_gold_table_name}"

        logging.info("Selected the req columns for the label data table")
        label_data = self.data.select(["feature_uid", "total_demand"])

        logging.info(f"Writing the label data. Path: {label_table_path}")
        label_data.write.format("delta")\
                        .mode("overwrite")\
                        .saveAsTable(label_table_path)
        logging.info(f"Label data saved at {label_table_path}")
        

In [0]:
featurestore = FeatureStore()

featurestore.read_gold_table()
featurestore.add_primary_key()
featurestore.register_feature_table()
featurestore.save_label_data()

In [0]:
%sql
--SELECT * FROM supply_chain_daily_demand.features.daily_demand_features

In [0]:
%sql
--SELECT COUNT(*) FROM supply_chain_daily_demand.features.daily_demand_features

In [0]:
%sql
--SELECT COUNT(*) FROM supply_chain_daily_demand.gold.daily_demand_label

In [0]:
%sql
--SELECT * FROM supply_chain_daily_demand.gold.daily_demand_label